# Feature Engineering and Target Construction

This notebook builds the final processed dataset used by the modeling notebook. Intermediate tables are built in memory for validation, but only `data/processed/feature_dataset.csv` is saved as the modeling artifact.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    BASE_MODEL_FEATURES,
    CLEAN_FINANCE_START,
    FEATURE_DATASET_PATH,
    MAX_INTRADAY_HORIZON_HOURS,
    MAX_OVERNIGHT_HORIZON_HOURS,
    PROCESSED_DATA_DIR,
    ROLLING_FEATURE_WINDOW,
    TARGET_RETURN_THRESHOLD,
)
from src.data_loading import load_merged_data
from src.feature_engineering import (
    add_next_close_target,
    build_clean_hourly_dataset,
    build_feature_dataset,
    build_hybrid_target_dataset,
)
from src.plots import set_plot_style

set_plot_style()
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

df = load_merged_data()
display(df.head())


Ticker,Timestamp,Text,Sentiment,Post_Count,Open,High,Low,Close,Volume
str,"datetime[μs, UTC]",str,f64,i64,f64,f64,f64,f64,i64
"""$AAPL""",2025-07-07 23:00:00 UTC,"""BREAKING: Apple’s, $AAPL, top …",-0.220868,2,210.130005,210.490005,208.449997,209.279999,9381795
"""$AAPL""",2025-07-07 23:00:00 UTC,"""🔴 Importance: 65/100 #news #To…",-0.440318,2,210.130005,210.490005,208.449997,209.279999,9381795
"""$AAPL""",2025-07-08 00:00:00 UTC,"""What do you think #investors? …",0.150104,2,210.130005,210.490005,208.449997,209.279999,9381795
"""$AAPL""",2025-07-08 00:00:00 UTC,"""Lemme guess... you bought AAPL…",0.012221,2,210.130005,210.490005,208.449997,209.279999,9381795
"""$AAPL""",2025-07-08 02:00:00 UTC,"""$AAPL Support / Resistance Lev…",0.027583,1,210.130005,210.490005,208.449997,209.279999,9381795


## Baseline Target Audit

This is the target used in the original notebook. We keep this audit cell to show why the post-level baseline target is not the final modeling target.

In [2]:
baseline_df = add_next_close_target(df)

print("Post-level target balance:")
print(baseline_df["Target_Direction"].value_counts().sort("Target_Direction"))


Post-level target balance:
shape: (2, 2)
┌──────────────────┬────────┐
│ Target_Direction ┆ count  │
│ ---              ┆ ---    │
│ i32              ┆ u32    │
╞══════════════════╪════════╡
│ 0                ┆ 187033 │
│ 1                ┆ 7835   │
└──────────────────┴────────┘


## Step 1: Clean Ticker-Hour Modeling Input

The raw dataset has one row per post. The final pipeline aggregates to one row per ticker-hour and filters out timestamps before the reliable Yahoo Finance hourly coverage window.

In [3]:
clean_hourly_df = build_clean_hourly_dataset(df)

print(f"Finance coverage cutoff: {CLEAN_FINANCE_START:%Y-%m-%d}")
print(f"Raw rows: {df.height:,}")
print(f"Clean ticker-hour rows: {clean_hourly_df.height:,}")
print(f"Unique tickers: {clean_hourly_df.select('Ticker').n_unique()}")
display(clean_hourly_df.head())


Finance coverage cutoff: 2024-06-01
Raw rows: 194,891
Clean ticker-hour rows: 54,572
Unique tickers: 23


Ticker,Timestamp,Open,High,Low,Close,Volume,Sentiment_Mean,Sentiment_Median,Sentiment_Std,Sentiment_Min,Sentiment_Max,Positive_Post_Count,Negative_Post_Count,Neutral_Post_Count,Post_Count,Bullishness_Index
str,"datetime[μs, UTC]",f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,i64,i64,i64,u32,f64
"""$AAPL""",2025-07-07 23:00:00 UTC,210.130005,210.490005,208.449997,209.279999,9381795,-0.330593,-0.330593,0.155175,-0.440318,-0.220868,0,2,0,2,-1.0
"""$AAPL""",2025-07-08 00:00:00 UTC,210.130005,210.490005,208.449997,209.279999,9381795,0.081162,0.081162,0.097498,0.012221,0.150104,1,0,1,2,0.5
"""$AAPL""",2025-07-08 02:00:00 UTC,210.130005,210.490005,208.449997,209.279999,9381795,0.027583,0.027583,0.0,0.027583,0.027583,0,0,1,1,0.0
"""$AAPL""",2025-07-08 05:00:00 UTC,210.130005,210.490005,208.449997,209.279999,9381795,-0.033409,0.075754,0.479426,-0.558003,0.382022,2,1,0,3,0.333333
"""$AAPL""",2025-07-08 06:00:00 UTC,210.130005,210.490005,208.449997,209.279999,9381795,0.019368,0.019368,0.0,0.019368,0.019368,0,0,1,1,0.0


## Step 1 Validation

In [4]:
duplicate_ticker_hours = clean_hourly_df.height - clean_hourly_df.select(["Ticker", "Timestamp"]).unique().height

print(f"Duplicate ticker-hour rows: {duplicate_ticker_hours}")
print("Timestamp range:")
display(clean_hourly_df.select(pl.col("Timestamp").min().alias("min_ts"), pl.col("Timestamp").max().alias("max_ts")))

print("Null counts:")
print(clean_hourly_df.null_count())

print("Bullishness index bounds:")
display(clean_hourly_df.select(pl.col("Bullishness_Index").min().alias("min"), pl.col("Bullishness_Index").max().alias("max")))


Duplicate ticker-hour rows: 0
Timestamp range:


min_ts,max_ts
"datetime[μs, UTC]","datetime[μs, UTC]"
2024-06-01 15:00:00 UTC,2026-04-28 23:00:00 UTC


Null counts:
shape: (1, 17)
┌────────┬───────────┬──────┬──────┬───┬───────────────┬───────────────┬────────────┬──────────────┐
│ Ticker ┆ Timestamp ┆ Open ┆ High ┆ … ┆ Negative_Post ┆ Neutral_Post_ ┆ Post_Count ┆ Bullishness_ │
│ ---    ┆ ---       ┆ ---  ┆ ---  ┆   ┆ _Count        ┆ Count         ┆ ---        ┆ Index        │
│ u32    ┆ u32       ┆ u32  ┆ u32  ┆   ┆ ---           ┆ ---           ┆ u32        ┆ ---          │
│        ┆           ┆      ┆      ┆   ┆ u32           ┆ u32           ┆            ┆ u32          │
╞════════╪═══════════╪══════╪══════╪═══╪═══════════════╪═══════════════╪════════════╪══════════════╡
│ 0      ┆ 0         ┆ 0    ┆ 0    ┆ … ┆ 0             ┆ 0             ┆ 0          ┆ 0            │
└────────┴───────────┴──────┴──────┴───┴───────────────┴───────────────┴────────────┴──────────────┘
Bullishness index bounds:


min,max
f64,f64
-1.0,1.0


## Step 2: Hybrid Intraday/Overnight Target

The final target is thresholded and tunable:

- `Target_Direction = 1` when future return is greater than `TARGET_RETURN_THRESHOLD`.
- `Target_Direction = 0` when future return is less than `-TARGET_RETURN_THRESHOLD`.
- Tiny moves between those thresholds are dropped as noisy neutral moves.

Hybrid target logic:

- **Intraday rows** use regular-session sentiment at hour `t` to predict the next observed same-day market bar.
- **Overnight rows** aggregate off-market sentiment before the next observed market open and predict the gap from the previous observed market close to that open.

In [5]:
hybrid_df = build_hybrid_target_dataset(
    clean_hourly_df,
    threshold=TARGET_RETURN_THRESHOLD,
    max_intraday_horizon_hours=MAX_INTRADAY_HORIZON_HOURS,
    max_overnight_horizon_hours=MAX_OVERNIGHT_HORIZON_HOURS,
    drop_neutral=True,
)

print(f"Return threshold: +/-{TARGET_RETURN_THRESHOLD:.4%}")
print(f"Max intraday target horizon: {MAX_INTRADAY_HORIZON_HOURS} hours")
print(f"Max overnight target horizon: {MAX_OVERNIGHT_HORIZON_HOURS} hours")
print(f"Hybrid target rows: {hybrid_df.height:,}")
display(hybrid_df.head())


Return threshold: +/-0.1000%
Max intraday target horizon: 6 hours
Max overnight target horizon: 96 hours
Hybrid target rows: 13,325


Target_Type,Ticker,Timestamp,Signal_Start,Signal_End,Target_Timestamp,Reference_Timestamp,Target_Horizon_Hours,Open,High,Low,Close,Volume,Sentiment_Mean,Sentiment_Median,Sentiment_Std,Sentiment_Min,Sentiment_Max,Positive_Post_Count,Negative_Post_Count,Neutral_Post_Count,Post_Count,Bullishness_Index,Reference_Price,Target_Price,Target_Return,Target_Direction
str,str,"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]",f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,i64,i64,i64,i64,f64,f64,f64,f64,i8
"""intraday""","""$AAPL""",2025-07-08 13:00:00 UTC,2025-07-08 13:00:00 UTC,2025-07-08 13:00:00 UTC,2025-07-08 14:00:00 UTC,2025-07-08 13:00:00 UTC,1.0,210.130005,210.490005,208.449997,209.279999,9381795,-0.218546,-0.006071,0.411388,-0.692725,0.043158,0,1,2,3,-0.333333,209.279999,209.870697,0.002823,1
"""overnight""","""$AAPL""",2025-07-09 12:00:00 UTC,2025-07-08 20:00:00 UTC,2025-07-09 12:00:00 UTC,2025-07-09 13:00:00 UTC,2025-07-08 19:00:00 UTC,1.0,210.035004,210.035004,210.035004,210.035004,5121140,0.043206,0.028212,0.402097,-0.937532,0.923812,9,3,5,17,0.352941,210.035004,209.529999,-0.002404,0
"""intraday""","""$AAPL""",2025-07-09 13:00:00 UTC,2025-07-09 13:00:00 UTC,2025-07-09 13:00:00 UTC,2025-07-09 14:00:00 UTC,2025-07-09 13:00:00 UTC,1.0,209.529999,210.899994,209.350006,209.910004,8992949,-0.367082,-0.304557,0.439264,-0.852405,0.045089,0,3,3,6,-0.5,209.910004,207.804993,-0.010028,0
"""intraday""","""$AAPL""",2025-07-09 15:00:00 UTC,2025-07-09 15:00:00 UTC,2025-07-09 15:00:00 UTC,2025-07-09 16:00:00 UTC,2025-07-09 15:00:00 UTC,1.0,207.809998,208.220001,207.2659,207.800003,4676493,0.397451,0.254352,0.477901,0.007448,0.930554,2,0,1,3,0.666667,207.800003,208.915604,0.005369,1
"""intraday""","""$AAPL""",2025-07-09 16:00:00 UTC,2025-07-09 16:00:00 UTC,2025-07-09 16:00:00 UTC,2025-07-09 17:00:00 UTC,2025-07-09 16:00:00 UTC,1.0,207.810394,209.044998,207.571396,208.915604,3964765,0.030129,0.030129,0.020721,0.015476,0.044781,0,0,2,2,0.0,208.915604,209.732498,0.00391,1


## Step 2 Validation

In [6]:
print("Rows by target type and direction:")
display(hybrid_df.group_by(["Target_Type", "Target_Direction"]).len().sort(["Target_Type", "Target_Direction"]))

print("Return summary:")
display(
    hybrid_df.group_by("Target_Type").agg([
        pl.col("Target_Return").min().alias("min"),
        pl.col("Target_Return").quantile(0.25).alias("p25"),
        pl.col("Target_Return").median().alias("median"),
        pl.col("Target_Return").quantile(0.75).alias("p75"),
        pl.col("Target_Return").max().alias("max"),
    ]).sort("Target_Type")
)

print("Target horizon summary:")
display(
    hybrid_df.group_by("Target_Type").agg([
        pl.col("Target_Horizon_Hours").min().alias("min_h"),
        pl.col("Target_Horizon_Hours").median().alias("median_h"),
        pl.col("Target_Horizon_Hours").max().alias("max_h"),
    ]).sort("Target_Type")
)


Rows by target type and direction:


Target_Type,Target_Direction,len
str,i8,u32
"""intraday""",0,4995
"""intraday""",1,5001
"""overnight""",0,1587
"""overnight""",1,1742


Return summary:


Target_Type,min,p25,median,p75,max
str,f64,f64,f64,f64,f64
"""intraday""",-0.223698,-0.004648,0.001006,0.004695,0.200241
"""overnight""",-0.332619,-0.00978,0.001671,0.011567,0.445334


Target horizon summary:


Target_Type,min_h,median_h,max_h
str,f64,f64,f64
"""intraday""",1.0,1.0,6.0
"""overnight""",1.0,2.0,88.0


## Step 3: Final Feature Dataset

This step adds model-ready predictors:

- Financial history: hourly return, close return lags, range, log volume, and volume anomaly.
- Sentiment history: short/long EMA, rolling z-score, post-count anomaly, and bullishness.
- Calendar features: cyclic hour and weekday features.
- Model split support: `Target_Type`, `Is_Intraday`, `Is_Overnight`, and ticker indicator columns.

`Target_Return` and price target columns are kept for evaluation and explanation, but they must be excluded from model features.

In [7]:
feature_df = build_feature_dataset(
    hybrid_df,
    clean_hourly_df,
    rolling_window=ROLLING_FEATURE_WINDOW,
)

print(f"Final feature rows: {feature_df.height:,}")
print(f"Final feature columns: {len(feature_df.columns):,}")
print(f"Base model features before ticker dummies: {len(BASE_MODEL_FEATURES):,}")
display(feature_df.head())


Final feature rows: 13,325
Final feature columns: 70
Base model features before ticker dummies: 32


Target_Type,Ticker,Timestamp,Signal_Start,Signal_End,Target_Timestamp,Reference_Timestamp,Target_Horizon_Hours,Open,High,Low,Close,Volume,Sentiment_Mean,Sentiment_Median,Sentiment_Std,Sentiment_Min,Sentiment_Max,Positive_Post_Count,Negative_Post_Count,Neutral_Post_Count,Post_Count,Bullishness_Index,Reference_Price,Target_Price,Target_Return,Target_Direction,Hourly_Return,High_Low_Range,Log_Volume,Close_Return_1,Close_Return_6,Close_Return_24,Sentiment_EMA_4,Sentiment_EMA_24,Volume_Z_24,Sentiment_Z_24,Post_Count_Z_24,Signal_Hour,Signal_Weekday,Signal_Month,Is_Overnight,Is_Intraday,Signal_Hour_Sin,Signal_Hour_Cos,Signal_Weekday_Sin,Signal_Weekday_Cos,Ticker_AAPL,Ticker_AMC,Ticker_AMD,Ticker_AMZN,Ticker_ARM,Ticker_BABA,Ticker_COIN,Ticker_DJT,Ticker_GME,Ticker_GOOGL,Ticker_HOOD,Ticker_INTC,Ticker_META,Ticker_MSFT,Ticker_MSTR,Ticker_NFLX,Ticker_NVDA,Ticker_PLTR,Ticker_QQQ,Ticker_RDDT,Ticker_SMCI,Ticker_SPY,Ticker_TSLA
str,str,"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]",f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,i64,i64,i64,i64,f64,f64,f64,f64,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8
"""intraday""","""$AAPL""",2025-07-08 13:00:00 UTC,2025-07-08 13:00:00 UTC,2025-07-08 13:00:00 UTC,2025-07-08 14:00:00 UTC,2025-07-08 13:00:00 UTC,1.0,210.130005,210.490005,208.449997,209.279999,9381795,-0.218546,-0.006071,0.411388,-0.692725,0.043158,0,1,2,3,-0.333333,209.279999,209.870697,0.002823,1,-0.004045,0.009708,16.054282,0.0,0.0,0.0,-0.082006,-0.180233,0.0,0.0,0.0,9,2,7,0,1,0.707107,-0.707107,0.974928,-0.222521,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""overnight""","""$AAPL""",2025-07-09 12:00:00 UTC,2025-07-08 20:00:00 UTC,2025-07-09 12:00:00 UTC,2025-07-09 13:00:00 UTC,2025-07-08 19:00:00 UTC,1.0,210.035004,210.035004,210.035004,210.035004,5121140,0.043206,0.028212,0.402097,-0.937532,0.923812,9,3,5,17,0.352941,210.035004,209.529999,-0.002404,0,0.002123,0.004962,15.448888,0.0,0.0,0.0,0.081355,-0.108015,0.0,0.0,0.0,8,3,7,1,0,0.866025,-0.5,0.433884,-0.900969,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""intraday""","""$AAPL""",2025-07-09 13:00:00 UTC,2025-07-09 13:00:00 UTC,2025-07-09 13:00:00 UTC,2025-07-09 14:00:00 UTC,2025-07-09 13:00:00 UTC,1.0,209.529999,210.899994,209.350006,209.910004,8992949,-0.367082,-0.304557,0.439264,-0.852405,0.045089,0,3,3,6,-0.5,209.910004,207.804993,-0.010028,0,0.001814,0.007397,16.011951,-0.000595,-0.000595,0.0,-0.09802,-0.12874,0.0,0.0,0.0,9,3,7,0,1,0.707107,-0.707107,0.433884,-0.900969,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""intraday""","""$AAPL""",2025-07-09 15:00:00 UTC,2025-07-09 15:00:00 UTC,2025-07-09 15:00:00 UTC,2025-07-09 16:00:00 UTC,2025-07-09 15:00:00 UTC,1.0,207.809998,208.220001,207.2659,207.800003,4676493,0.397451,0.254352,0.477901,0.007448,0.930554,2,0,1,3,0.666667,207.800003,208.915604,0.005369,1,-0.000048,0.004591,15.358059,-0.000024,-0.010641,-0.007072,0.13368,-0.074107,-1.034587,1.579918,0.4552,11,3,7,0,1,0.258819,-0.965926,0.433884,-0.900969,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""intraday""","""$AAPL""",2025-07-09 16:00:00 UTC,2025-07-09 16:00:00 UTC,2025-07-09 16:00:00 UTC,2025-07-09 17:00:00 UTC,2025-07-09 16:00:00 UTC,1.0,207.810394,209.044998,207.571396,208.915604,3964765,0.030129,0.030129,0.020721,0.015476,0.044781,0,0,2,2,0.0,208.915604,209.732498,0.00391,1,0.005318,0.007091,15.192957,0.005369,-0.00533,-0.001741,0.09226,-0.065768,-1.260291,0.304927,-0.255297,12,3,7,0,1,1.2246e-16,-1.0,0.433884,-0.900969,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Step 3 Validation

In [8]:
ticker_dummy_cols = [col for col in feature_df.columns if col.startswith("Ticker_")]
missing_base_features = [col for col in BASE_MODEL_FEATURES if col not in feature_df.columns]

print("Rows by target type and direction:")
display(feature_df.group_by(["Target_Type", "Target_Direction"]).len().sort(["Target_Type", "Target_Direction"]))

print("Ticker dummy count:", len(ticker_dummy_cols))
print("Missing base model features:", missing_base_features)
print("Null total:", feature_df.null_count().sum_horizontal().item())
print("Duplicate signal/target rows:", feature_df.height - feature_df.select(["Ticker", "Timestamp", "Target_Type", "Target_Timestamp"]).unique().height)

print("Selected engineered feature summary:")
display(
    feature_df.select([
        pl.col("Close_Return_1").mean().alias("mean_close_return_1"),
        pl.col("Close_Return_24").mean().alias("mean_close_return_24"),
        pl.col("Volume_Z_24").mean().alias("mean_volume_z_24"),
        pl.col("Sentiment_Z_24").mean().alias("mean_sentiment_z_24"),
        pl.col("Post_Count_Z_24").mean().alias("mean_post_count_z_24"),
    ])
)


Rows by target type and direction:


Target_Type,Target_Direction,len
str,i8,u32
"""intraday""",0,4995
"""intraday""",1,5001
"""overnight""",0,1587
"""overnight""",1,1742


Ticker dummy count: 23
Missing base model features: []
Null total: 0
Duplicate signal/target rows: 0
Selected engineered feature summary:


mean_close_return_1,mean_close_return_24,mean_volume_z_24,mean_sentiment_z_24,mean_post_count_z_24
f64,f64,f64,f64,f64
0.000298,0.003417,0.398204,0.00016,0.505524


## Correlation Audit

We explicitly check pairwise correlations among the main numeric modeling features. The OHLC price columns are expected to be highly correlated because they describe the same hourly bar. We keep them in the final feature set because the final models are chosen for prediction, the tree models are robust to redundant predictors, and the logistic-regression baseline uses regularization. For interpretation, we rely more on engineered return/range/anomaly features than raw OHLC coefficients.

In [9]:
corr_feature_cols = [
    "Open", "High", "Low", "Close", "Volume", "Log_Volume",
    "Hourly_Return", "High_Low_Range", "Close_Return_1", "Close_Return_6", "Close_Return_24",
    "Volume_Z_24", "Sentiment_Mean", "Sentiment_EMA_4", "Sentiment_EMA_24", "Sentiment_Z_24",
    "Post_Count", "Post_Count_Z_24", "Bullishness_Index",
]

corr_matrix = feature_df.select(corr_feature_cols).corr()
corr_pairs = []
for i, feature_1 in enumerate(corr_feature_cols):
    for j, feature_2 in enumerate(corr_feature_cols):
        if j <= i:
            continue
        corr_pairs.append({
            "feature_1": feature_1,
            "feature_2": feature_2,
            "corr": corr_matrix[feature_1][j],
        })

corr_pairs_df = (
    pl.DataFrame(corr_pairs)
    .with_columns(pl.col("corr").abs().alias("abs_corr"))
    .sort("abs_corr", descending=True)
)

high_corr_pairs = corr_pairs_df.filter(pl.col("abs_corr") >= 0.95)
print(f"Pairs with |corr| >= 0.95: {high_corr_pairs.height}")
print(high_corr_pairs.with_columns(pl.col(["corr", "abs_corr"]).round(6)).to_pandas().to_string(index=False))


Pairs with |corr| >= 0.95: 6
feature_1 feature_2     corr  abs_corr
      Low     Close 0.999952  0.999952
     Open      High 0.999948  0.999948
     Open       Low 0.999943  0.999943
     High     Close 0.999942  0.999942
     Open     Close 0.999925  0.999925
     High       Low 0.999879  0.999879


## Preprocessing Rationale, Bias Controls, and Tradeoffs

This section makes the preprocessing choices explicit because the rubric asks for justification, limitations, and awareness of conceptual errors.

- **Why ticker-hour aggregation?** The raw file has one row per post, so a single highly discussed hour can appear many times with the same future price movement. Modeling at the post level would overweight viral hours and create a misleading target balance. Aggregating to one ticker-hour row makes each observed ticker-hour contribute once while preserving post count, sentiment dispersion, and bullishness as features.
- **Why the June 1, 2024 finance cutoff?** Yahoo Finance hourly coverage is limited to recent history. Some raw Bluesky timestamps are much older, so keeping all raw timestamps would mix reliable hourly bars with filled or stale market context. The cutoff prioritizes valid finance coverage over maximum row count.
- **Why the hybrid intraday/overnight target?** Market-hour sentiment and off-market sentiment answer different questions. Intraday rows ask whether sentiment during an open market hour predicts the next same-day bar. Overnight rows ask whether off-market discussion predicts the next opening gap. Combining both while marking `Target_Type` lets the model learn shared structure without hiding this domain difference.
- **Why threshold returns and drop neutral moves?** Directional labels from tiny returns are noisy and can flip because of bid/ask effects, microstructure, and random hourly movement. The +/-0.1% threshold makes the target more meaningful. The tradeoff is that the final labeled set is smaller and no longer represents all market hours.
- **Null handling justification:** Rows missing core market or sentiment fields are removed before aggregation because those values define the predictive unit. Later engineered numeric nulls from rolling windows and lags are filled with 0 only after feature construction; in this context 0 means no measured anomaly or no prior lag information at the beginning of a ticker history.
- **Outlier handling justification:** We avoid removing high-volume or extreme-return rows outright because spikes are meaningful in financial/social data. Instead, we use log volume, rolling z-scores, clipping inside `_safe_z_score`, and a direction target so that large values remain informative without dominating purely by scale.
- **Correlation and multicollinearity:** The correlation audit confirms that OHLC prices are highly correlated. This is expected, so we document it rather than silently ignoring it. We keep these fields because the final models are prediction-oriented and tree models tolerate redundant predictors; however, interpretation focuses on return/range/anomaly features rather than raw OHLC coefficients.
- **Imbalance handling timing:** Class imbalance is addressed in the modeling notebook only after train/validation/test splitting. This avoids the leakage penalty described in the rubric, because synthetic or duplicated training examples are never allowed to influence validation or test rows.
- **Remaining bias:** These features can describe observed associations, not causal impact. Social-media discussion may react to price movement rather than cause it, and ticker popularity can act as a proxy for unobserved news intensity.

## Save Final Modeling Artifact

This is the only processed CSV the modeling notebook should load.

In [10]:
feature_df.write_csv(FEATURE_DATASET_PATH)
print(f"Saved {feature_df.height:,} rows to {FEATURE_DATASET_PATH}")


Saved 13,325 rows to C:\Users\dpnim\Documents\cis 2450\cis-2450-final-proj\data\processed\feature_dataset.csv


## Next Work

- Update `03_modeling_and_results.ipynb` to load `data/processed/feature_dataset.csv`.
- Train both a combined model and separate intraday/overnight models.
- Apply scaling and any imbalance handling only after the train/test split.